In [1]:
import re
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = (
    Path.home()
    / "projects"
    / "chart_data_retrieval"
)

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
BENCHMARK_DIR = PROJECT_ROOT / "data" / "benchmark"

QUERIES_PATH = PROCESSED_DIR / "queries.parquet"
PILOT_PATH = BENCHMARK_DIR / "pilot_tables.parquet"
GENERATED_CHARTS_PATH = (
    BENCHMARK_DIR / "generated_charts.parquet"
)

# Confirm required files
for path in [
    QUERIES_PATH,
    PILOT_PATH,
    GENERATED_CHARTS_PATH,
]:
    print(path.name, ":", path.exists())
    assert path.exists(), f"Missing file: {path}"

# Load benchmark files
queries_df = pd.read_parquet(QUERIES_PATH)
pilot_df = pd.read_parquet(PILOT_PATH)
generated_charts_df = pd.read_parquet(
    GENERATED_CHARTS_PATH
)

print("\nAll ChartQA queries:", len(queries_df))
print("Pilot tables:", len(pilot_df))
print("Generated charts:", len(generated_charts_df))

assert len(pilot_df) == 700
assert len(generated_charts_df) == 4104

# Keep only queries belonging to the 700 pilot tables
pilot_data_ids = set(pilot_df["data_id"])

pilot_queries_df = queries_df[
    queries_df["data_id"].isin(pilot_data_ids)
].copy()

# Questions explicitly tied to a visual style should not
# automatically treat all chart types as equivalent.
style_pattern = re.compile(
    r"\b("
    r"bars?|bar[\s-]?graph|"
    r"lines?|line[\s-]?graph|"
    r"pie(?:\s+chart)?|slices?|"
    r"legend|colou?r|"
    r"x[\s-]?axis|y[\s-]?axis|axes?"
    r")\b",
    flags=re.IGNORECASE,
)

pilot_queries_df["is_style_sensitive"] = (
    pilot_queries_df["question"]
    .str.contains(
        style_pattern,
        regex=True,
        na=False,
    )
)

pilot_queries_df["query_scope"] = np.where(
    pilot_queries_df["is_style_sensitive"],
    "style_sensitive",
    "data_semantic",
)

data_queries_df = pilot_queries_df[
    pilot_queries_df["query_scope"] == "data_semantic"
].copy()

print("\nPilot tables with at least one question:")
print(pilot_queries_df["data_id"].nunique())

print("\nAll pilot queries:")
print(len(pilot_queries_df))

print("\nQueries by scope:")
print(
    pilot_queries_df["query_scope"].value_counts()
)

print("\nUsable data-semantic queries by split:")
print(data_queries_df["split"].value_counts())

print("\nQueries by source:")
print(data_queries_df["question_source"].value_counts())

display(
    pilot_queries_df[
        [
            "query_id",
            "data_id",
            "split",
            "question",
            "answer",
            "query_scope",
        ]
    ].head(10)
)

queries.parquet : True
pilot_tables.parquet : True
generated_charts.parquet : True

All ChartQA queries: 32719
Pilot tables: 700
Generated charts: 4104

Pilot tables with at least one question:
700

All pilot queries:
1066

Queries by scope:
query_scope
data_semantic      986
style_sensitive     80
Name: count, dtype: int64

Usable data-semantic queries by split:
split
train    689
val      151
test     146
Name: count, dtype: int64

Queries by source:
question_source
augmented    782
human        204
Name: count, dtype: int64


/tmp/ipykernel_2578564/1104244703.py:67: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  .str.contains(


,query_id,data_id,split,question,answer,query_scope
586,chartqa_train_human_586,chartqa_train_11993,train,What is the value of smallest section of the p...,9,style_sensitive
587,chartqa_train_human_587,chartqa_train_11993,train,What is the difference between largest pie sec...,49,style_sensitive
592,chartqa_train_human_592,chartqa_train_12127,train,What is the percentage value of Job related in...,0.42,style_sensitive
593,chartqa_train_human_593,chartqa_train_12127,train,Is the sum of Other and Job related more then ...,No,data_semantic
608,chartqa_train_human_608,chartqa_train_12582,train,How many percentage British now want to remain...,0.5,data_semantic
609,chartqa_train_human_609,chartqa_train_12582,train,How many percentage people want to leave EU?,0.41,data_semantic
610,chartqa_train_human_610,chartqa_train_12604,train,What percent does Mostly agree represent?,33,data_semantic
611,chartqa_train_human_611,chartqa_train_12604,train,"Is the sum of Mostly disagree ,Completely disa...",No,data_semantic
612,chartqa_train_human_612,chartqa_train_12626,train,What does 8 percent represent?,Don't know,data_semantic
613,chartqa_train_human_613,chartqa_train_12626,train,What is the difference in value of Here to sta...,6,data_semantic


In [2]:
import hashlib
import json
from pathlib import Path

import pandas as pd


# --------------------------------------------------
# 1. Prepare chart metadata
# --------------------------------------------------

charts_for_queries_df = generated_charts_df.copy()

charts_for_queries_df["is_hard_negative"] = (
    charts_for_queries_df["is_hard_negative"]
    .fillna(False)
    .astype(bool)
)

# Add number of categories to every generated chart.
pilot_metadata_df = (
    pilot_df[
        [
            "data_id",
            "num_rows",
        ]
    ]
    .drop_duplicates("data_id")
    .rename(
        columns={
            "data_id": "source_data_id",
            "num_rows": "source_num_rows",
        }
    )
)

charts_for_queries_df = (
    charts_for_queries_df
    .merge(
        pilot_metadata_df,
        on="source_data_id",
        how="left",
        validate="many_to_one",
    )
)

assert (
    charts_for_queries_df["source_num_rows"]
    .notna()
    .all()
)

# Charts containing the original data are possible positives.
positive_chart_pool_df = charts_for_queries_df[
    ~charts_for_queries_df["is_hard_negative"]
].copy()

# Value-shuffled charts remain in the retrieval index,
# but they are never marked as correct answers.
counterfactual_chart_pool_df = charts_for_queries_df[
    charts_for_queries_df["is_hard_negative"]
].copy()

print(
    "Positive chart candidates:",
    len(positive_chart_pool_df),
)

print(
    "Counterfactual chart candidates:",
    len(counterfactual_chart_pool_df),
)


# --------------------------------------------------
# 2. Create evaluation ground truth
# --------------------------------------------------

ground_truth_records = []

for _, query_row in data_queries_df.iterrows():
    query_data_id = query_row["data_id"]

    matching_positive_charts = (
        positive_chart_pool_df[
            positive_chart_pool_df["source_data_id"]
            == query_data_id
        ]
        .sort_values("chart_id")
    )

    matching_counterfactual_charts = (
        counterfactual_chart_pool_df[
            counterfactual_chart_pool_df["source_data_id"]
            == query_data_id
        ]
        .sort_values("chart_id")
    )

    positive_chart_ids = (
        matching_positive_charts["chart_id"]
        .tolist()
    )

    counterfactual_chart_ids = (
        matching_counterfactual_charts["chart_id"]
        .tolist()
    )

    assert len(positive_chart_ids) >= 3
    assert len(counterfactual_chart_ids) >= 2

    ground_truth_records.append({
        "query_id": query_row["query_id"],
        "data_id": query_data_id,
        "source_data_id": query_data_id,
        "split": query_row["split"],
        "question_source": query_row["question_source"],
        "question": query_row["question"],
        "answer": query_row["answer"],
        "query_scope": "data_semantic",
        "positive_chart_ids_json": json.dumps(
            positive_chart_ids
        ),
        "counterfactual_chart_ids_json": json.dumps(
            counterfactual_chart_ids
        ),
        "num_positive_charts": len(
            positive_chart_ids
        ),
        "num_counterfactual_charts": len(
            counterfactual_chart_ids
        ),
    })

query_chart_ground_truth_df = pd.DataFrame(
    ground_truth_records
)


# --------------------------------------------------
# 3. Deterministic negative selection
# --------------------------------------------------

def deterministic_index(text, length):
    digest = hashlib.sha256(
        text.encode("utf-8")
    ).hexdigest()

    return int(digest[:16], 16) % length


# --------------------------------------------------
# 4. Create query→chart contrastive triplets
# --------------------------------------------------

query_chart_pair_records = []

for _, query_row in data_queries_df.iterrows():
    query_id = query_row["query_id"]
    query_data_id = query_row["data_id"]
    query_split = query_row["split"]

    positive_charts = (
        positive_chart_pool_df[
            positive_chart_pool_df["source_data_id"]
            == query_data_id
        ]
        .sort_values("chart_id")
    )

    for _, positive_chart in positive_charts.iterrows():
        positive_style = positive_chart["style_id"]
        positive_num_rows = int(
            positive_chart["source_num_rows"]
        )

        # Same split and same visual style,
        # but a completely different source table.
        negative_candidates = (
            positive_chart_pool_df[
                (
                    positive_chart_pool_df["split"]
                    == query_split
                )
                & (
                    positive_chart_pool_df["style_id"]
                    == positive_style
                )
                & (
                    positive_chart_pool_df["source_data_id"]
                    != query_data_id
                )
            ]
            .copy()
        )

        assert len(negative_candidates) > 0, (
            f"No negative candidates for "
            f"{query_id}, style={positive_style}"
        )

        # Make the negative visually harder by preferring
        # charts with a similar number of categories.
        negative_candidates["row_difference"] = (
            negative_candidates["source_num_rows"]
            .sub(positive_num_rows)
            .abs()
        )

        smallest_difference = (
            negative_candidates["row_difference"]
            .min()
        )

        hardest_candidates = (
            negative_candidates[
                negative_candidates["row_difference"]
                == smallest_difference
            ]
            .sort_values("chart_id")
            .reset_index(drop=True)
        )

        selected_index = deterministic_index(
            (
                f"{query_id}__"
                f"{positive_chart['chart_id']}"
            ),
            len(hardest_candidates),
        )

        negative_chart = hardest_candidates.iloc[
            selected_index
        ]

        query_chart_pair_records.append({
            "pair_id": (
                f"{query_id}__"
                f"{positive_chart['chart_id']}"
            ),
            "task": "query_chart",
            "split": query_split,
            "anchor_id": query_id,
            "positive_id": positive_chart["chart_id"],
            "negative_id": negative_chart["chart_id"],
            "anchor_modality": "query",
            "positive_modality": "chart",
            "negative_modality": "chart",
            "source_data_id": query_data_id,
            "negative_source_data_id": (
                negative_chart["source_data_id"]
            ),
            "negative_type": (
                "same_style_different_data"
            ),
            "question_source": (
                query_row["question_source"]
            ),
            "question": query_row["question"],
            "answer": query_row["answer"],
            "query_scope": "data_semantic",
            "positive_chart_type": (
                positive_chart["chart_type"]
            ),
            "negative_chart_type": (
                negative_chart["chart_type"]
            ),
            "positive_style_id": positive_style,
            "negative_style_id": (
                negative_chart["style_id"]
            ),
            "positive_variant_type": (
                positive_chart["variant_type"]
            ),
            "negative_variant_type": (
                negative_chart["variant_type"]
            ),
            "positive_image_path": (
                positive_chart["image_path"]
            ),
            "negative_image_path": (
                negative_chart["image_path"]
            ),
            "positive_num_rows": positive_num_rows,
            "negative_num_rows": int(
                negative_chart["source_num_rows"]
            ),
        })

query_chart_pairs_df = pd.DataFrame(
    query_chart_pair_records
)


# --------------------------------------------------
# 5. Validate the benchmark
# --------------------------------------------------

assert query_chart_pairs_df["pair_id"].is_unique
assert query_chart_ground_truth_df["query_id"].is_unique

assert (
    query_chart_pairs_df["positive_style_id"]
    == query_chart_pairs_df["negative_style_id"]
).all()

assert (
    query_chart_pairs_df["source_data_id"]
    != query_chart_pairs_df[
        "negative_source_data_id"
    ]
).all()

assert (
    query_chart_pairs_df["anchor_id"].nunique()
    == len(data_queries_df)
)

known_chart_ids = set(
    generated_charts_df["chart_id"]
)

assert set(
    query_chart_pairs_df["positive_id"]
).issubset(known_chart_ids)

assert set(
    query_chart_pairs_df["negative_id"]
).issubset(known_chart_ids)

missing_images = [
    path
    for path in generated_charts_df["image_path"]
    if not Path(path).exists()
]

assert not missing_images, (
    f"Missing chart images: {len(missing_images)}"
)


# --------------------------------------------------
# 6. Save outputs
# --------------------------------------------------

QUERY_CHART_PAIRS_PATH = (
    BENCHMARK_DIR
    / "query_chart_pairs.parquet"
)

QUERY_CHART_GROUND_TRUTH_PATH = (
    BENCHMARK_DIR
    / "query_chart_ground_truth.parquet"
)

STYLE_SENSITIVE_QUERIES_PATH = (
    BENCHMARK_DIR
    / "style_sensitive_queries.parquet"
)

query_chart_pairs_df.to_parquet(
    QUERY_CHART_PAIRS_PATH,
    index=False,
)

query_chart_ground_truth_df.to_parquet(
    QUERY_CHART_GROUND_TRUTH_PATH,
    index=False,
)

pilot_queries_df[
    pilot_queries_df["query_scope"]
    == "style_sensitive"
].to_parquet(
    STYLE_SENSITIVE_QUERIES_PATH,
    index=False,
)


# --------------------------------------------------
# 7. Report
# --------------------------------------------------

print("\nQuery→chart triplets:", len(query_chart_pairs_df))

print(
    "Unique usable queries:",
    query_chart_pairs_df["anchor_id"].nunique(),
)

print(
    "\nTriplets by split:"
)

print(
    query_chart_pairs_df["split"].value_counts()
)

print(
    "\nPositive chart types:"
)

print(
    query_chart_pairs_df[
        "positive_chart_type"
    ].value_counts()
)

print(
    "\nPositive variants:"
)

print(
    query_chart_pairs_df[
        "positive_variant_type"
    ].value_counts()
)

print("\nSaved files:")

for path in [
    QUERY_CHART_PAIRS_PATH,
    QUERY_CHART_GROUND_TRUTH_PATH,
    STYLE_SENSITIVE_QUERIES_PATH,
]:
    print(
        path.name,
        ":",
        path.exists(),
    )

Positive chart candidates: 2402
Counterfactual chart candidates: 1702

Query→chart triplets: 3413
Unique usable queries: 986

Triplets by split:
split
train    2370
val       525
test      518
Name: count, dtype: int64

Positive chart types:
positive_chart_type
h_bar    986
line     986
v_bar    986
pie      455
Name: count, dtype: int64

Positive variants:
positive_variant_type
hard_positive    1441
positive          986
anchor            986
Name: count, dtype: int64

Saved files:
query_chart_pairs.parquet : True
query_chart_ground_truth.parquet : True
style_sensitive_queries.parquet : True


In [3]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from transformers import (
    CLIPModel,
    CLIPProcessor,
)


# --------------------------------------------------
# Project paths
# --------------------------------------------------

PROJECT_ROOT = (
    Path.home()
    / "projects"
    / "chart_data_retrieval"
)

BENCHMARK_DIR = PROJECT_ROOT / "data" / "benchmark"

EMBEDDING_DIR = (
    PROJECT_ROOT
    / "artifacts"
    / "embeddings"
    / "clip_vit_b32"
)

MODEL_CACHE_DIR = (
    PROJECT_ROOT
    / "artifacts"
    / "model_cache"
)

EMBEDDING_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

MODEL_CACHE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# --------------------------------------------------
# Load benchmark metadata
# --------------------------------------------------

CHARTS_PATH = (
    BENCHMARK_DIR
    / "generated_charts.parquet"
)

QUERY_GROUND_TRUTH_PATH = (
    BENCHMARK_DIR
    / "query_chart_ground_truth.parquet"
)

QUERY_PAIRS_PATH = (
    BENCHMARK_DIR
    / "query_chart_pairs.parquet"
)

for path in [
    CHARTS_PATH,
    QUERY_GROUND_TRUTH_PATH,
    QUERY_PAIRS_PATH,
]:
    print(path.name, ":", path.exists())
    assert path.exists(), f"Missing file: {path}"

charts_df = (
    pd.read_parquet(CHARTS_PATH)
    .sort_values("chart_id")
    .reset_index(drop=True)
)

queries_df = (
    pd.read_parquet(QUERY_GROUND_TRUTH_PATH)
    .sort_values("query_id")
    .reset_index(drop=True)
)

query_pairs_df = pd.read_parquet(
    QUERY_PAIRS_PATH
)

assert len(charts_df) == 4104
assert len(queries_df) == 986
assert len(query_pairs_df) == 3413

print("\nCharts:", len(charts_df))
print("Queries:", len(queries_df))
print("Query triplets:", len(query_pairs_df))


# --------------------------------------------------
# GPU and model
# --------------------------------------------------

DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

MODEL_NAME = "openai/clip-vit-base-patch32"

print("\nDevice:", DEVICE)

if DEVICE == "cuda":
    print(
        "GPU:",
        torch.cuda.get_device_name(0),
    )

processor = CLIPProcessor.from_pretrained(
    MODEL_NAME,
    cache_dir=MODEL_CACHE_DIR,
)

model = CLIPModel.from_pretrained(
    MODEL_NAME,
    cache_dir=MODEL_CACHE_DIR,
)

model = model.to(DEVICE)
model.eval()

print("Model loaded:", MODEL_NAME)
print(
    "Embedding dimension:",
    model.config.projection_dim,
)

generated_charts.parquet : True
query_chart_ground_truth.parquet : True
query_chart_pairs.parquet : True

Charts: 4104
Queries: 986
Query triplets: 3413

Device: cuda
GPU: NVIDIA A100-SXM4-80GB MIG 2g.20gb


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

/home/pnola130/.conda/envs/chart_retrieval/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

Model loaded: openai/clip-vit-base-patch32
Embedding dimension: 512


In [4]:
print("CUDA available:", torch.cuda.is_available())
print("Model device:", next(model.parameters()).device)
print(
    "GPU memory used:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)

CUDA available: True
Model device: cuda:0
GPU memory used: 0.56 GB


In [5]:
import json

import numpy as np
import torch
import torch.nn.functional as F

from PIL import Image
from tqdm.auto import tqdm


IMAGE_BATCH_SIZE = 64
TEXT_BATCH_SIZE = 128


def load_rgb_image(image_path):
    with Image.open(image_path) as image:
        return image.convert("RGB")


# --------------------------------------------------
# 1. Encode chart images
# --------------------------------------------------

chart_embedding_batches = []

for start in tqdm(
    range(0, len(charts_df), IMAGE_BATCH_SIZE),
    desc="Encoding chart images",
):
    end = min(
        start + IMAGE_BATCH_SIZE,
        len(charts_df),
    )

    batch_paths = charts_df.iloc[
        start:end
    ]["image_path"].tolist()

    batch_images = [
        load_rgb_image(path)
        for path in batch_paths
    ]

    image_inputs = processor(
        images=batch_images,
        return_tensors="pt",
    )

    pixel_values = image_inputs[
        "pixel_values"
    ].to(DEVICE)

    with torch.inference_mode():
        image_features = model.get_image_features(
            pixel_values=pixel_values
        )

        image_features = F.normalize(
            image_features,
            p=2,
            dim=1,
        )

    chart_embedding_batches.append(
        image_features
        .float()
        .cpu()
        .numpy()
    )

chart_embeddings = np.concatenate(
    chart_embedding_batches,
    axis=0,
).astype("float32")


# --------------------------------------------------
# 2. Encode query text
# --------------------------------------------------

query_embedding_batches = []

for start in tqdm(
    range(0, len(queries_df), TEXT_BATCH_SIZE),
    desc="Encoding query text",
):
    end = min(
        start + TEXT_BATCH_SIZE,
        len(queries_df),
    )

    batch_questions = queries_df.iloc[
        start:end
    ]["question"].astype(str).tolist()

    text_inputs = processor(
        text=batch_questions,
        padding=True,
        truncation=True,
        max_length=77,
        return_tensors="pt",
    )

    text_inputs = {
        key: value.to(DEVICE)
        for key, value in text_inputs.items()
    }

    with torch.inference_mode():
        text_features = model.get_text_features(
            **text_inputs
        )

        text_features = F.normalize(
            text_features,
            p=2,
            dim=1,
        )

    query_embedding_batches.append(
        text_features
        .float()
        .cpu()
        .numpy()
    )

query_embeddings = np.concatenate(
    query_embedding_batches,
    axis=0,
).astype("float32")


# --------------------------------------------------
# 3. Validate embeddings
# --------------------------------------------------

assert chart_embeddings.shape == (
    4104,
    512,
)

assert query_embeddings.shape == (
    986,
    512,
)

assert np.isfinite(chart_embeddings).all()
assert np.isfinite(query_embeddings).all()

chart_norms = np.linalg.norm(
    chart_embeddings,
    axis=1,
)

query_norms = np.linalg.norm(
    query_embeddings,
    axis=1,
)

assert np.allclose(
    chart_norms,
    1.0,
    atol=1e-4,
)

assert np.allclose(
    query_norms,
    1.0,
    atol=1e-4,
)


# --------------------------------------------------
# 4. Create embedding-index metadata
# --------------------------------------------------

chart_embedding_metadata_df = charts_df.copy()

chart_embedding_metadata_df.insert(
    0,
    "embedding_index",
    np.arange(len(chart_embedding_metadata_df)),
)

query_embedding_metadata_df = queries_df.copy()

query_embedding_metadata_df.insert(
    0,
    "embedding_index",
    np.arange(len(query_embedding_metadata_df)),
)


# --------------------------------------------------
# 5. Save embeddings and metadata
# --------------------------------------------------

CHART_EMBEDDINGS_PATH = (
    EMBEDDING_DIR
    / "chart_embeddings.npy"
)

QUERY_EMBEDDINGS_PATH = (
    EMBEDDING_DIR
    / "query_embeddings.npy"
)

CHART_METADATA_PATH = (
    EMBEDDING_DIR
    / "chart_metadata.parquet"
)

QUERY_METADATA_PATH = (
    EMBEDDING_DIR
    / "query_metadata.parquet"
)

CONFIG_PATH = (
    EMBEDDING_DIR
    / "embedding_config.json"
)

np.save(
    CHART_EMBEDDINGS_PATH,
    chart_embeddings,
)

np.save(
    QUERY_EMBEDDINGS_PATH,
    query_embeddings,
)

chart_embedding_metadata_df.to_parquet(
    CHART_METADATA_PATH,
    index=False,
)

query_embedding_metadata_df.to_parquet(
    QUERY_METADATA_PATH,
    index=False,
)

embedding_config = {
    "model_name": MODEL_NAME,
    "embedding_dimension": 512,
    "normalized": True,
    "number_of_charts": len(charts_df),
    "number_of_queries": len(queries_df),
    "image_batch_size": IMAGE_BATCH_SIZE,
    "text_batch_size": TEXT_BATCH_SIZE,
}

with open(
    CONFIG_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        embedding_config,
        file,
        indent=2,
    )


# --------------------------------------------------
# 6. Report
# --------------------------------------------------

print("\nChart embeddings:", chart_embeddings.shape)
print("Query embeddings:", query_embeddings.shape)

print(
    "Mean chart norm:",
    round(float(chart_norms.mean()), 6),
)

print(
    "Mean query norm:",
    round(float(query_norms.mean()), 6),
)

print("\nSaved files:")

for path in [
    CHART_EMBEDDINGS_PATH,
    QUERY_EMBEDDINGS_PATH,
    CHART_METADATA_PATH,
    QUERY_METADATA_PATH,
    CONFIG_PATH,
]:
    print(
        path.name,
        ":",
        path.exists(),
    )

Encoding chart images:   0%|          | 0/65 [00:00<?, ?it/s]

Encoding query text:   0%|          | 0/8 [00:00<?, ?it/s]


Chart embeddings: (4104, 512)
Query embeddings: (986, 512)
Mean chart norm: 1.0
Mean query norm: 1.0

Saved files:
chart_embeddings.npy : True
query_embeddings.npy : True
chart_metadata.parquet : True
query_metadata.parquet : True
embedding_config.json : True


In [6]:
import json
from pathlib import Path

import faiss
import numpy as np
import pandas as pd


# --------------------------------------------------
# 1. Output directories
# --------------------------------------------------

INDEX_DIR = (
    PROJECT_ROOT
    / "artifacts"
    / "indexes"
    / "clip_vit_b32"
)

METRICS_DIR = (
    PROJECT_ROOT
    / "reports"
    / "metrics"
)

INDEX_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

METRICS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# --------------------------------------------------
# 2. Verify embedding alignment
# --------------------------------------------------

assert (
    chart_embedding_metadata_df["chart_id"].tolist()
    == charts_df["chart_id"].tolist()
)

assert (
    query_embedding_metadata_df["query_id"].tolist()
    == queries_df["query_id"].tolist()
)

assert chart_embeddings.dtype == np.float32
assert query_embeddings.dtype == np.float32


# --------------------------------------------------
# 3. Metric helpers
# --------------------------------------------------

def calculate_ndcg(relevance, number_of_positives, k=10):
    relevance_at_k = relevance[:k]

    discounts = np.log2(
        np.arange(2, len(relevance_at_k) + 2)
    )

    dcg = np.sum(
        relevance_at_k / discounts
    )

    ideal_positive_count = min(
        number_of_positives,
        k,
    )

    if ideal_positive_count == 0:
        return 0.0

    ideal_relevance = np.ones(
        ideal_positive_count,
        dtype=np.float32,
    )

    ideal_discounts = np.log2(
        np.arange(2, ideal_positive_count + 2)
    )

    idcg = np.sum(
        ideal_relevance / ideal_discounts
    )

    return float(dcg / idcg)


def summarize_results(results):
    return {
        "number_of_queries": int(len(results)),
        "recall_at_1": float(
            results["recall_at_1"].mean()
        ),
        "recall_at_5": float(
            results["recall_at_5"].mean()
        ),
        "recall_at_10": float(
            results["recall_at_10"].mean()
        ),
        "mrr": float(
            results["reciprocal_rank"].mean()
        ),
        "ndcg_at_10": float(
            results["ndcg_at_10"].mean()
        ),
        "mean_first_positive_rank": float(
            results["first_positive_rank"].mean()
        ),
        "median_first_positive_rank": float(
            results["first_positive_rank"].median()
        ),
        "counterfactual_outrank_rate": float(
            results[
                "counterfactual_outranks_positive"
            ].mean()
        ),
        "mean_query_data_gap": float(
            results["query_data_gap"].mean()
        ),
    }


# --------------------------------------------------
# 4. Build one FAISS index per split
# --------------------------------------------------

result_records = []
candidate_counts = {}

for split in ["train", "val", "test"]:
    chart_indices = np.flatnonzero(
        charts_df["split"].to_numpy() == split
    )

    query_indices = np.flatnonzero(
        queries_df["split"].to_numpy() == split
    )

    split_chart_embeddings = np.ascontiguousarray(
        chart_embeddings[chart_indices],
        dtype=np.float32,
    )

    split_query_embeddings = np.ascontiguousarray(
        query_embeddings[query_indices],
        dtype=np.float32,
    )

    split_chart_metadata = (
        charts_df
        .iloc[chart_indices]
        .reset_index(drop=True)
        .copy()
    )

    split_query_metadata = (
        queries_df
        .iloc[query_indices]
        .reset_index(drop=True)
        .copy()
    )

    candidate_counts[split] = len(
        split_chart_metadata
    )

    # Inner product equals cosine similarity because
    # all embeddings were L2-normalized.
    index = faiss.IndexFlatIP(
        split_chart_embeddings.shape[1]
    )

    index.add(split_chart_embeddings)

    assert index.ntotal == len(
        split_chart_metadata
    )

    # Search the complete split to obtain exact MRR.
    similarities, local_indices = index.search(
        split_query_embeddings,
        len(split_chart_metadata),
    )

    candidate_chart_ids = (
        split_chart_metadata["chart_id"]
        .to_numpy()
    )

    for query_position, query_row in (
        split_query_metadata.iterrows()
    ):
        retrieved_chart_ids = candidate_chart_ids[
            local_indices[query_position]
        ]

        retrieved_similarities = similarities[
            query_position
        ]

        positive_chart_ids = set(
            json.loads(
                query_row[
                    "positive_chart_ids_json"
                ]
            )
        )

        counterfactual_chart_ids = set(
            json.loads(
                query_row[
                    "counterfactual_chart_ids_json"
                ]
            )
        )

        relevance = np.array(
            [
                chart_id in positive_chart_ids
                for chart_id in retrieved_chart_ids
            ],
            dtype=np.float32,
        )

        counterfactual_relevance = np.array(
            [
                chart_id in counterfactual_chart_ids
                for chart_id in retrieved_chart_ids
            ],
            dtype=np.float32,
        )

        positive_positions = np.flatnonzero(
            relevance
        )

        counterfactual_positions = np.flatnonzero(
            counterfactual_relevance
        )

        assert len(positive_positions) > 0
        assert len(counterfactual_positions) > 0

        first_positive_rank = int(
            positive_positions[0] + 1
        )

        first_counterfactual_rank = int(
            counterfactual_positions[0] + 1
        )

        best_positive_similarity = float(
            retrieved_similarities[
                positive_positions
            ].max()
        )

        best_counterfactual_similarity = float(
            retrieved_similarities[
                counterfactual_positions
            ].max()
        )

        query_data_gap = (
            best_positive_similarity
            - best_counterfactual_similarity
        )

        result_records.append({
            "query_id": query_row["query_id"],
            "data_id": query_row["data_id"],
            "split": split,
            "question_source": (
                query_row["question_source"]
            ),
            "question": query_row["question"],
            "answer": query_row["answer"],
            "number_of_candidates": len(
                split_chart_metadata
            ),
            "number_of_positives": len(
                positive_chart_ids
            ),
            "first_positive_rank": (
                first_positive_rank
            ),
            "first_counterfactual_rank": (
                first_counterfactual_rank
            ),
            "recall_at_1": (
                first_positive_rank <= 1
            ),
            "recall_at_5": (
                first_positive_rank <= 5
            ),
            "recall_at_10": (
                first_positive_rank <= 10
            ),
            "reciprocal_rank": (
                1.0 / first_positive_rank
            ),
            "ndcg_at_10": calculate_ndcg(
                relevance,
                len(positive_chart_ids),
                k=10,
            ),
            "best_positive_similarity": (
                best_positive_similarity
            ),
            "best_counterfactual_similarity": (
                best_counterfactual_similarity
            ),
            "query_data_gap": query_data_gap,
            "counterfactual_outranks_positive": (
                first_counterfactual_rank
                < first_positive_rank
            ),
            "top_1_chart_id": (
                retrieved_chart_ids[0]
            ),
            "top_1_similarity": float(
                retrieved_similarities[0]
            ),
            "top_10_chart_ids_json": json.dumps(
                retrieved_chart_ids[:10].tolist()
            ),
        })

    # Save the index and its ID mapping.
    index_path = (
        INDEX_DIR
        / f"charts_{split}.faiss"
    )

    mapping_path = (
        INDEX_DIR
        / f"charts_{split}_mapping.parquet"
    )

    faiss.write_index(
        index,
        str(index_path),
    )

    split_chart_mapping = (
        split_chart_metadata.copy()
    )

    split_chart_mapping.insert(
        0,
        "faiss_index",
        np.arange(len(split_chart_mapping)),
    )

    split_chart_mapping.to_parquet(
        mapping_path,
        index=False,
    )

    print(
        f"{split}: "
        f"{len(split_query_metadata)} queries, "
        f"{len(split_chart_metadata)} candidates"
    )


# --------------------------------------------------
# 5. Aggregate results
# --------------------------------------------------

query_chart_results_df = pd.DataFrame(
    result_records
)

assert len(query_chart_results_df) == 986

overall_metrics = summarize_results(
    query_chart_results_df
)

split_metrics = {
    split: summarize_results(
        query_chart_results_df[
            query_chart_results_df["split"]
            == split
        ]
    )
    for split in ["train", "val", "test"]
}

metrics_report = {
    "model_name": MODEL_NAME,
    "embedding_dimension": 512,
    "similarity": "cosine",
    "candidate_counts": candidate_counts,
    "overall": overall_metrics,
    "by_split": split_metrics,
}


# --------------------------------------------------
# 6. Save results
# --------------------------------------------------

RESULTS_PATH = (
    METRICS_DIR
    / "clip_query_chart_results.parquet"
)

METRICS_PATH = (
    METRICS_DIR
    / "clip_query_chart_metrics.json"
)

query_chart_results_df.to_parquet(
    RESULTS_PATH,
    index=False,
)

with open(
    METRICS_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        metrics_report,
        file,
        indent=2,
    )


# --------------------------------------------------
# 7. Display metrics
# --------------------------------------------------

metrics_table = pd.DataFrame(
    {
        "overall": overall_metrics,
        **split_metrics,
    }
).T

display(
    metrics_table[
        [
            "number_of_queries",
            "recall_at_1",
            "recall_at_5",
            "recall_at_10",
            "mrr",
            "ndcg_at_10",
            "median_first_positive_rank",
            "counterfactual_outrank_rate",
            "mean_query_data_gap",
        ]
    ]
)

print("\nSaved:", RESULTS_PATH)
print("Saved:", METRICS_PATH)

for split in ["train", "val", "test"]:
    print(
        split,
        "index exists:",
        (
            INDEX_DIR
            / f"charts_{split}.faiss"
        ).exists(),
    )

train: 689 queries, 2918 candidates
val: 151 queries, 592 candidates
test: 146 queries, 594 candidates


,number_of_queries,recall_at_1,recall_at_5,recall_at_10,mrr,ndcg_at_10,median_first_positive_rank,counterfactual_outrank_rate,mean_query_data_gap
overall,986.0,0.249493,0.467546,0.558824,0.360079,0.290616,7.0,0.288032,0.005882
train,689.0,0.198839,0.417997,0.512337,0.310599,0.249030,9.0,0.301887,0.005958
val,151.0,0.350993,0.556291,0.662252,0.457874,0.371761,4.0,0.251656,0.006181
test,146.0,0.383562,0.609589,0.671233,0.492435,0.402944,2.0,0.260274,0.005212



Saved: /home/pnola130/projects/chart_data_retrieval/reports/metrics/clip_query_chart_results.parquet
Saved: /home/pnola130/projects/chart_data_retrieval/reports/metrics/clip_query_chart_metrics.json
train index exists: True
val index exists: True
test index exists: True


In [7]:
import json

import numpy as np
import pandas as pd


# --------------------------------------------------
# 1. Load chart→chart triplets
# --------------------------------------------------

RETRIEVAL_PAIRS_PATH = (
    BENCHMARK_DIR
    / "retrieval_pairs.parquet"
)

retrieval_pairs_df = pd.read_parquet(
    RETRIEVAL_PAIRS_PATH
)

chart_chart_pairs_df = (
    retrieval_pairs_df[
        retrieval_pairs_df["task"]
        == "chart_chart"
    ]
    .copy()
    .reset_index(drop=True)
)

print(
    "Chart→chart triplets:",
    len(chart_chart_pairs_df),
)

assert len(chart_chart_pairs_df) == 2704


# --------------------------------------------------
# 2. Map chart IDs to embedding rows
# --------------------------------------------------

chart_id_to_embedding_index = dict(
    zip(
        charts_df["chart_id"],
        range(len(charts_df)),
    )
)

for column in [
    "anchor_id",
    "positive_id",
    "negative_id",
]:
    chart_chart_pairs_df[
        f"{column}_embedding_index"
    ] = chart_chart_pairs_df[column].map(
        chart_id_to_embedding_index
    )

    assert chart_chart_pairs_df[
        f"{column}_embedding_index"
    ].notna().all()


anchor_indices = chart_chart_pairs_df[
    "anchor_id_embedding_index"
].astype(int).to_numpy()

positive_indices = chart_chart_pairs_df[
    "positive_id_embedding_index"
].astype(int).to_numpy()

negative_indices = chart_chart_pairs_df[
    "negative_id_embedding_index"
].astype(int).to_numpy()


# --------------------------------------------------
# 3. Calculate cosine similarities
# --------------------------------------------------

anchor_embeddings = chart_embeddings[
    anchor_indices
]

positive_embeddings = chart_embeddings[
    positive_indices
]

negative_embeddings = chart_embeddings[
    negative_indices
]

anchor_positive_similarity = np.sum(
    anchor_embeddings * positive_embeddings,
    axis=1,
)

anchor_negative_similarity = np.sum(
    anchor_embeddings * negative_embeddings,
    axis=1,
)

similarity_gap = (
    anchor_positive_similarity
    - anchor_negative_similarity
)

chart_chart_results_df = (
    chart_chart_pairs_df.copy()
)

chart_chart_results_df[
    "same_data_different_style_similarity"
] = anchor_positive_similarity

chart_chart_results_df[
    "same_style_different_data_similarity"
] = anchor_negative_similarity

chart_chart_results_df[
    "style_data_similarity_gap"
] = similarity_gap

chart_chart_results_df[
    "data_preferred_over_style"
] = similarity_gap > 0

chart_chart_results_df[
    "gap_above_0_05"
] = similarity_gap >= 0.05


# --------------------------------------------------
# 4. Add chart metadata
# --------------------------------------------------

chart_metadata_by_id = charts_df.set_index(
    "chart_id"
)

metadata_columns = {
    "chart_type": "chart_type",
    "style_id": "style_id",
    "data_id": "data_id",
    "variant_type": "variant_type",
}

for role in [
    "anchor",
    "positive",
    "negative",
]:
    id_column = f"{role}_id"

    for source_column, output_suffix in (
        metadata_columns.items()
    ):
        chart_chart_results_df[
            f"{role}_{output_suffix}"
        ] = chart_chart_results_df[
            id_column
        ].map(
            chart_metadata_by_id[
                source_column
            ]
        )

chart_chart_results_df["pair_family"] = (
    chart_chart_results_df[
        "anchor_chart_type"
    ].astype(str)
    + "_to_"
    + chart_chart_results_df[
        "positive_chart_type"
    ].astype(str)
)


# --------------------------------------------------
# 5. Validate benchmark semantics
# --------------------------------------------------

# Anchor and positive have the same underlying values.
assert (
    chart_chart_results_df["anchor_data_id"]
    == chart_chart_results_df["positive_data_id"]
).all()

# Negative has shuffled/different data.
assert (
    chart_chart_results_df["anchor_data_id"]
    != chart_chart_results_df["negative_data_id"]
).all()

# Anchor and negative share the same visual style.
assert (
    chart_chart_results_df["anchor_style_id"]
    == chart_chart_results_df["negative_style_id"]
).all()

# Positive changes the visual style.
assert (
    chart_chart_results_df["anchor_style_id"]
    != chart_chart_results_df["positive_style_id"]
).all()


# --------------------------------------------------
# 6. Summarize results
# --------------------------------------------------

def summarize_chart_chart(group):
    return pd.Series({
        "number_of_triplets": len(group),
        "mean_positive_similarity": group[
            "same_data_different_style_similarity"
        ].mean(),
        "mean_negative_similarity": group[
            "same_style_different_data_similarity"
        ].mean(),
        "mean_similarity_gap": group[
            "style_data_similarity_gap"
        ].mean(),
        "median_similarity_gap": group[
            "style_data_similarity_gap"
        ].median(),
        "data_over_style_accuracy": group[
            "data_preferred_over_style"
        ].mean(),
        "style_bias_rate": (
            ~group["data_preferred_over_style"]
        ).mean(),
        "gap_above_0_05_rate": group[
            "gap_above_0_05"
        ].mean(),
    })


overall_chart_chart_metrics = (
    summarize_chart_chart(
        chart_chart_results_df
    )
)

family_chart_chart_metrics = (
    chart_chart_results_df
    .groupby("pair_family", sort=True)
    .apply(summarize_chart_chart)
)

chart_chart_metrics_table = pd.concat(
    [
        overall_chart_chart_metrics
        .to_frame()
        .T
        .rename(index={0: "overall"}),

        family_chart_chart_metrics,
    ]
)

display(chart_chart_metrics_table)


# --------------------------------------------------
# 7. Save results
# --------------------------------------------------

CHART_CHART_RESULTS_PATH = (
    METRICS_DIR
    / "clip_chart_chart_results.parquet"
)

CHART_CHART_METRICS_PATH = (
    METRICS_DIR
    / "clip_chart_chart_metrics.json"
)

chart_chart_results_df.to_parquet(
    CHART_CHART_RESULTS_PATH,
    index=False,
)

chart_chart_metrics_report = {
    "model_name": MODEL_NAME,
    "definition": (
        "similarity(same data, different style) "
        "- similarity(same style, different data)"
    ),
    "overall": {
        key: (
            int(value)
            if key == "number_of_triplets"
            else float(value)
        )
        for key, value
        in overall_chart_chart_metrics.items()
    },
    "by_pair_family": {
        family: {
            key: (
                int(value)
                if key == "number_of_triplets"
                else float(value)
            )
            for key, value in row.items()
        }
        for family, row
        in family_chart_chart_metrics.iterrows()
    },
}

with open(
    CHART_CHART_METRICS_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        chart_chart_metrics_report,
        file,
        indent=2,
    )

print("\nSaved:", CHART_CHART_RESULTS_PATH)
print("Saved:", CHART_CHART_METRICS_PATH)

Chart→chart triplets: 2704


/tmp/ipykernel_2578564/2597083418.py:248: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(summarize_chart_chart)


,number_of_triplets,mean_positive_similarity,mean_negative_similarity,mean_similarity_gap,median_similarity_gap,data_over_style_accuracy,style_bias_rate,gap_above_0_05_rate
overall,2704.0,0.852413,0.978242,-0.125829,-0.098957,0.002219,0.997781,0.000370
line_to_v_bar,700.0,0.898921,0.973896,-0.074975,-0.071550,0.000000,1.000000,0.000000
pie_to_v_bar,302.0,0.716655,0.904121,-0.187466,-0.194256,0.019868,0.980132,0.003311
v_bar_to_h_bar,700.0,0.876536,0.992907,-0.116371,-0.104497,0.000000,1.000000,0.000000
v_bar_to_line,700.0,0.898921,0.992907,-0.093985,-0.091098,0.000000,1.000000,0.000000
v_bar_to_pie,302.0,0.716655,0.994455,-0.277800,-0.274675,0.000000,1.000000,0.000000



Saved: /home/pnola130/projects/chart_data_retrieval/reports/metrics/clip_chart_chart_results.parquet
Saved: /home/pnola130/projects/chart_data_retrieval/reports/metrics/clip_chart_chart_metrics.json


In [8]:
import json

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from tqdm.auto import tqdm


# --------------------------------------------------
# 1. Load benchmark tables
# --------------------------------------------------

BENCHMARK_TABLES_PATH = (
    BENCHMARK_DIR
    / "benchmark_tables.parquet"
)

benchmark_tables_df = (
    pd.read_parquet(BENCHMARK_TABLES_PATH)
    .sort_values("data_id")
    .reset_index(drop=True)
)

assert len(benchmark_tables_df) == 1400
assert benchmark_tables_df["data_id"].is_unique

print("Benchmark tables:", len(benchmark_tables_df))

print(
    "\nTable variants:"
)

print(
    benchmark_tables_df[
        "variant_type"
    ].value_counts()
)


# --------------------------------------------------
# 2. Measure CLIP text truncation
# --------------------------------------------------

table_token_counts = []

for table_text in tqdm(
    benchmark_tables_df["table_text"].astype(str),
    desc="Counting table tokens",
):
    token_ids = processor.tokenizer.encode(
        table_text,
        add_special_tokens=True,
        truncation=False,
    )

    table_token_counts.append(
        len(token_ids)
    )

benchmark_tables_df[
    "clip_token_count"
] = table_token_counts

benchmark_tables_df[
    "clip_was_truncated"
] = (
    benchmark_tables_df["clip_token_count"]
    > 77
)

print(
    "\nMean table tokens:",
    round(
        benchmark_tables_df[
            "clip_token_count"
        ].mean(),
        2,
    ),
)

print(
    "Maximum table tokens:",
    benchmark_tables_df[
        "clip_token_count"
    ].max(),
)

print(
    "Tables truncated:",
    benchmark_tables_df[
        "clip_was_truncated"
    ].sum(),
)

print(
    "Truncation rate:",
    round(
        benchmark_tables_df[
            "clip_was_truncated"
        ].mean(),
        4,
    ),
)


# --------------------------------------------------
# 3. Encode table text
# --------------------------------------------------

TABLE_TEXT_BATCH_SIZE = 128
table_embedding_batches = []

for start in tqdm(
    range(
        0,
        len(benchmark_tables_df),
        TABLE_TEXT_BATCH_SIZE,
    ),
    desc="Encoding table text",
):
    end = min(
        start + TABLE_TEXT_BATCH_SIZE,
        len(benchmark_tables_df),
    )

    batch_table_text = (
        benchmark_tables_df
        .iloc[start:end]["table_text"]
        .astype(str)
        .tolist()
    )

    table_inputs = processor(
        text=batch_table_text,
        padding=True,
        truncation=True,
        max_length=77,
        return_tensors="pt",
    )

    table_inputs = {
        key: value.to(DEVICE)
        for key, value in table_inputs.items()
    }

    with torch.inference_mode():
        table_features = model.get_text_features(
            **table_inputs
        )

        table_features = F.normalize(
            table_features,
            p=2,
            dim=1,
        )

    table_embedding_batches.append(
        table_features
        .float()
        .cpu()
        .numpy()
    )

table_embeddings = np.concatenate(
    table_embedding_batches,
    axis=0,
).astype("float32")


# --------------------------------------------------
# 4. Validate
# --------------------------------------------------

assert table_embeddings.shape == (
    1400,
    512,
)

assert np.isfinite(
    table_embeddings
).all()

table_norms = np.linalg.norm(
    table_embeddings,
    axis=1,
)

assert np.allclose(
    table_norms,
    1.0,
    atol=1e-4,
)


# --------------------------------------------------
# 5. Save embeddings and metadata
# --------------------------------------------------

table_embedding_metadata_df = (
    benchmark_tables_df.copy()
)

table_embedding_metadata_df.insert(
    0,
    "embedding_index",
    np.arange(
        len(table_embedding_metadata_df)
    ),
)

TABLE_EMBEDDINGS_PATH = (
    EMBEDDING_DIR
    / "table_embeddings.npy"
)

TABLE_METADATA_PATH = (
    EMBEDDING_DIR
    / "table_metadata.parquet"
)

np.save(
    TABLE_EMBEDDINGS_PATH,
    table_embeddings,
)

table_embedding_metadata_df.to_parquet(
    TABLE_METADATA_PATH,
    index=False,
)

# Update embedding configuration.
with open(
    CONFIG_PATH,
    "r",
    encoding="utf-8",
) as file:
    embedding_config = json.load(file)

embedding_config.update({
    "number_of_tables": len(
        benchmark_tables_df
    ),
    "table_text_max_tokens": 77,
    "table_text_truncation_rate": float(
        benchmark_tables_df[
            "clip_was_truncated"
        ].mean()
    ),
})

with open(
    CONFIG_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        embedding_config,
        file,
        indent=2,
    )


# --------------------------------------------------
# 6. Report
# --------------------------------------------------

print(
    "\nTable embeddings:",
    table_embeddings.shape,
)

print(
    "Mean table norm:",
    round(
        float(table_norms.mean()),
        6,
    ),
)

print(
    "table_embeddings.npy:",
    TABLE_EMBEDDINGS_PATH.exists(),
)

print(
    "table_metadata.parquet:",
    TABLE_METADATA_PATH.exists(),
)

Benchmark tables: 1400

Table variants:
variant_type
original         700
hard_negative    700
Name: count, dtype: int64


Counting table tokens:   0%|          | 0/1400 [00:00<?, ?it/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (95 > 77). Running this sequence through the model will result in indexing errors



Mean table tokens: 101.41
Maximum table tokens: 305
Tables truncated: 924
Truncation rate: 0.66


Encoding table text:   0%|          | 0/11 [00:00<?, ?it/s]


Table embeddings: (1400, 512)
Mean table norm: 1.0
table_embeddings.npy: True
table_metadata.parquet: True


In [9]:
import json

import faiss
import numpy as np
import pandas as pd


# --------------------------------------------------
# 1. Load chart→table pairs
# --------------------------------------------------

chart_table_pairs_df = (
    retrieval_pairs_df[
        retrieval_pairs_df["task"]
        == "chart_table"
    ]
    .copy()
    .reset_index(drop=True)
)

print(
    "Chart→table pairs:",
    len(chart_table_pairs_df),
)

assert len(chart_table_pairs_df) == 2402
assert chart_table_pairs_df["anchor_id"].is_unique


# --------------------------------------------------
# 2. Verify embedding alignment
# --------------------------------------------------

assert (
    table_embedding_metadata_df["data_id"].tolist()
    == benchmark_tables_df["data_id"].tolist()
)

chart_id_to_embedding_index = dict(
    zip(
        charts_df["chart_id"],
        range(len(charts_df)),
    )
)

table_id_to_embedding_index = dict(
    zip(
        benchmark_tables_df["data_id"],
        range(len(benchmark_tables_df)),
    )
)

chart_table_pairs_df[
    "chart_embedding_index"
] = chart_table_pairs_df["anchor_id"].map(
    chart_id_to_embedding_index
)

chart_table_pairs_df[
    "positive_table_embedding_index"
] = chart_table_pairs_df["positive_id"].map(
    table_id_to_embedding_index
)

chart_table_pairs_df[
    "negative_table_embedding_index"
] = chart_table_pairs_df["negative_id"].map(
    table_id_to_embedding_index
)

assert chart_table_pairs_df[
    "chart_embedding_index"
].notna().all()

assert chart_table_pairs_df[
    "positive_table_embedding_index"
].notna().all()

assert chart_table_pairs_df[
    "negative_table_embedding_index"
].notna().all()


# --------------------------------------------------
# 3. Add chart metadata
# --------------------------------------------------

chart_metadata_by_id = charts_df.set_index(
    "chart_id"
)

for column in [
    "chart_type",
    "style_id",
    "variant_type",
]:
    chart_table_pairs_df[column] = (
        chart_table_pairs_df["anchor_id"]
        .map(
            chart_metadata_by_id[column]
        )
    )


# --------------------------------------------------
# 4. Build table FAISS indexes and evaluate
# --------------------------------------------------

chart_table_result_records = []
table_candidate_counts = {}

for split in ["train", "val", "test"]:
    split_table_indices = np.flatnonzero(
        benchmark_tables_df[
            "split"
        ].to_numpy() == split
    )

    split_pair_indices = np.flatnonzero(
        chart_table_pairs_df[
            "split"
        ].to_numpy() == split
    )

    split_table_embeddings = (
        np.ascontiguousarray(
            table_embeddings[
                split_table_indices
            ],
            dtype=np.float32,
        )
    )

    split_table_metadata = (
        benchmark_tables_df
        .iloc[split_table_indices]
        .reset_index(drop=True)
        .copy()
    )

    split_pairs = (
        chart_table_pairs_df
        .iloc[split_pair_indices]
        .reset_index(drop=True)
        .copy()
    )

    split_chart_indices = (
        split_pairs[
            "chart_embedding_index"
        ]
        .astype(int)
        .to_numpy()
    )

    split_chart_embeddings = (
        np.ascontiguousarray(
            chart_embeddings[
                split_chart_indices
            ],
            dtype=np.float32,
        )
    )

    table_candidate_counts[split] = len(
        split_table_metadata
    )

    table_index = faiss.IndexFlatIP(
        table_embeddings.shape[1]
    )

    table_index.add(
        split_table_embeddings
    )

    similarities, local_indices = (
        table_index.search(
            split_chart_embeddings,
            len(split_table_metadata),
        )
    )

    candidate_table_ids = (
        split_table_metadata[
            "data_id"
        ].to_numpy()
    )

    for pair_position, pair_row in (
        split_pairs.iterrows()
    ):
        retrieved_table_ids = (
            candidate_table_ids[
                local_indices[pair_position]
            ]
        )

        retrieved_similarities = (
            similarities[pair_position]
        )

        positive_positions = np.flatnonzero(
            retrieved_table_ids
            == pair_row["positive_id"]
        )

        negative_positions = np.flatnonzero(
            retrieved_table_ids
            == pair_row["negative_id"]
        )

        assert len(positive_positions) == 1
        assert len(negative_positions) == 1

        positive_position = int(
            positive_positions[0]
        )

        negative_position = int(
            negative_positions[0]
        )

        positive_rank = positive_position + 1
        negative_rank = negative_position + 1

        positive_similarity = float(
            retrieved_similarities[
                positive_position
            ]
        )

        negative_similarity = float(
            retrieved_similarities[
                negative_position
            ]
        )

        table_data_gap = (
            positive_similarity
            - negative_similarity
        )

        if positive_rank <= 10:
            ndcg_at_10 = float(
                1.0 / np.log2(
                    positive_rank + 1
                )
            )
        else:
            ndcg_at_10 = 0.0

        chart_table_result_records.append({
            "pair_id": pair_row["pair_id"],
            "split": split,
            "chart_id": pair_row["anchor_id"],
            "positive_table_id": (
                pair_row["positive_id"]
            ),
            "negative_table_id": (
                pair_row["negative_id"]
            ),
            "chart_type": pair_row["chart_type"],
            "style_id": pair_row["style_id"],
            "variant_type": (
                pair_row["variant_type"]
            ),
            "number_of_candidates": len(
                split_table_metadata
            ),
            "positive_table_rank": positive_rank,
            "negative_table_rank": negative_rank,
            "recall_at_1": positive_rank <= 1,
            "recall_at_5": positive_rank <= 5,
            "recall_at_10": positive_rank <= 10,
            "reciprocal_rank": (
                1.0 / positive_rank
            ),
            "ndcg_at_10": ndcg_at_10,
            "positive_table_similarity": (
                positive_similarity
            ),
            "negative_table_similarity": (
                negative_similarity
            ),
            "table_data_similarity_gap": (
                table_data_gap
            ),
            "data_preferred_over_counterfactual": (
                table_data_gap > 0
            ),
            "counterfactual_outranks_positive": (
                negative_rank < positive_rank
            ),
            "top_1_table_id": (
                retrieved_table_ids[0]
            ),
            "top_10_table_ids_json": json.dumps(
                retrieved_table_ids[:10].tolist()
            ),
        })

    # Save split-specific FAISS table index.
    table_index_path = (
        INDEX_DIR
        / f"tables_{split}.faiss"
    )

    table_mapping_path = (
        INDEX_DIR
        / f"tables_{split}_mapping.parquet"
    )

    faiss.write_index(
        table_index,
        str(table_index_path),
    )

    split_table_mapping = (
        split_table_metadata.copy()
    )

    split_table_mapping.insert(
        0,
        "faiss_index",
        np.arange(
            len(split_table_mapping)
        ),
    )

    split_table_mapping.to_parquet(
        table_mapping_path,
        index=False,
    )

    print(
        f"{split}: "
        f"{len(split_pairs)} chart queries, "
        f"{len(split_table_metadata)} tables"
    )


# --------------------------------------------------
# 5. Summarize metrics
# --------------------------------------------------

chart_table_results_df = pd.DataFrame(
    chart_table_result_records
)

assert len(chart_table_results_df) == 2402


def summarize_chart_table(group):
    return pd.Series({
        "number_of_charts": len(group),
        "recall_at_1": group[
            "recall_at_1"
        ].mean(),
        "recall_at_5": group[
            "recall_at_5"
        ].mean(),
        "recall_at_10": group[
            "recall_at_10"
        ].mean(),
        "mrr": group[
            "reciprocal_rank"
        ].mean(),
        "ndcg_at_10": group[
            "ndcg_at_10"
        ].mean(),
        "median_positive_rank": group[
            "positive_table_rank"
        ].median(),
        "mean_positive_similarity": group[
            "positive_table_similarity"
        ].mean(),
        "mean_negative_similarity": group[
            "negative_table_similarity"
        ].mean(),
        "mean_table_data_gap": group[
            "table_data_similarity_gap"
        ].mean(),
        "data_over_counterfactual_accuracy": group[
            "data_preferred_over_counterfactual"
        ].mean(),
        "counterfactual_outrank_rate": group[
            "counterfactual_outranks_positive"
        ].mean(),
    })


overall_chart_table_metrics = (
    summarize_chart_table(
        chart_table_results_df
    )
)

split_chart_table_metrics = (
    chart_table_results_df
    .groupby("split", sort=True)
    .apply(summarize_chart_table)
)

type_chart_table_metrics = (
    chart_table_results_df
    .groupby("chart_type", sort=True)
    .apply(summarize_chart_table)
)

chart_table_metrics_table = pd.concat(
    [
        overall_chart_table_metrics
        .to_frame()
        .T
        .rename(index={0: "overall"}),

        split_chart_table_metrics.add_prefix(
            "split_"
        ),

        type_chart_table_metrics.add_prefix(
            "type_"
        ),
    ]
)

display(
    chart_table_metrics_table[
        [
            "number_of_charts",
            "recall_at_1",
            "recall_at_5",
            "recall_at_10",
            "mrr",
            "ndcg_at_10",
            "median_positive_rank",
            "mean_table_data_gap",
            "data_over_counterfactual_accuracy",
            "counterfactual_outrank_rate",
        ]
    ]
)


# --------------------------------------------------
# 6. Save results
# --------------------------------------------------

CHART_TABLE_RESULTS_PATH = (
    METRICS_DIR
    / "clip_chart_table_results.parquet"
)

CHART_TABLE_METRICS_PATH = (
    METRICS_DIR
    / "clip_chart_table_metrics.json"
)

chart_table_results_df.to_parquet(
    CHART_TABLE_RESULTS_PATH,
    index=False,
)

chart_table_metrics_report = {
    "model_name": MODEL_NAME,
    "table_encoder": (
        "CLIP text encoder over serialized table"
    ),
    "maximum_table_tokens": 77,
    "table_truncation_rate": float(
        benchmark_tables_df[
            "clip_was_truncated"
        ].mean()
    ),
    "candidate_counts": table_candidate_counts,
    "overall": {
        key: (
            int(value)
            if key == "number_of_charts"
            else float(value)
        )
        for key, value
        in overall_chart_table_metrics.items()
    },
    "by_split": {
        split: {
            key: (
                int(value)
                if key == "number_of_charts"
                else float(value)
            )
            for key, value in row.items()
        }
        for split, row
        in split_chart_table_metrics.iterrows()
    },
    "by_chart_type": {
        chart_type: {
            key: (
                int(value)
                if key == "number_of_charts"
                else float(value)
            )
            for key, value in row.items()
        }
        for chart_type, row
        in type_chart_table_metrics.iterrows()
    },
}

with open(
    CHART_TABLE_METRICS_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        chart_table_metrics_report,
        file,
        indent=2,
    )

print("\nSaved:", CHART_TABLE_RESULTS_PATH)
print("Saved:", CHART_TABLE_METRICS_PATH)

for split in ["train", "val", "test"]:
    print(
        split,
        "table index exists:",
        (
            INDEX_DIR
            / f"tables_{split}.faiss"
        ).exists(),
    )

Chart→table pairs: 2402
train: 1709 chart queries, 1000 tables
val: 346 chart queries, 200 tables
test: 347 chart queries, 200 tables


/tmp/ipykernel_2578564/2700509802.py:402: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(summarize_chart_table)
/tmp/ipykernel_2578564/2700509802.py:408: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(summarize_chart_table)


,number_of_charts,recall_at_1,recall_at_5,recall_at_10,mrr,ndcg_at_10,median_positive_rank,mean_table_data_gap,data_over_counterfactual_accuracy,counterfactual_outrank_rate
overall,2402.0,0.209409,0.620316,0.726062,0.402907,0.474474,3.0,-0.000098,0.468776,0.531224
test,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
train,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
val,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
h_bar,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
line,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
pie,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
v_bar,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Saved: /home/pnola130/projects/chart_data_retrieval/reports/metrics/clip_chart_table_results.parquet
Saved: /home/pnola130/projects/chart_data_retrieval/reports/metrics/clip_chart_table_metrics.json
train table index exists: True
val table index exists: True
test table index exists: True


In [10]:
import json
import pandas as pd

RESULTS_PATH = (
    METRICS_DIR
    / "clip_chart_table_results.parquet"
)

METRICS_PATH = (
    METRICS_DIR
    / "clip_chart_table_metrics.json"
)

# Reload the correctly saved per-chart results
chart_table_results_df = pd.read_parquet(RESULTS_PATH)

metric_rows = []

# Overall metrics
overall_metrics = summarize_chart_table(
    chart_table_results_df
).to_dict()

metric_rows.append({
    "group": "overall",
    **overall_metrics
})

# Metrics by split
for split_name, split_df in chart_table_results_df.groupby(
    "split",
    sort=False
):
    split_metrics = summarize_chart_table(
        split_df
    ).to_dict()

    metric_rows.append({
        "group": split_name,
        **split_metrics
    })

# Metrics by chart type
for chart_type, type_df in chart_table_results_df.groupby(
    "chart_type",
    sort=False
):
    type_metrics = summarize_chart_table(
        type_df
    ).to_dict()

    metric_rows.append({
        "group": chart_type,
        **type_metrics
    })

# Build the corrected table
chart_table_metrics_df = (
    pd.DataFrame(metric_rows)
    .set_index("group")
)

display(chart_table_metrics_df.round(6))

# Overwrite the incorrect JSON file
with open(METRICS_PATH, "w", encoding="utf-8") as file:
    json.dump(
        chart_table_metrics_df.to_dict(
            orient="index"
        ),
        file,
        indent=2
    )

print("Corrected metrics saved:", METRICS_PATH)


,number_of_charts,recall_at_1,recall_at_5,recall_at_10,mrr,ndcg_at_10,median_positive_rank,mean_positive_similarity,mean_negative_similarity,mean_table_data_gap,data_over_counterfactual_accuracy,counterfactual_outrank_rate
group,,,,,,,,,,,,
overall,2402.0,0.209409,0.620316,0.726062,0.402907,0.474474,3.0,0.321905,0.322003,-0.000098,0.468776,0.531224
train,1709.0,0.184903,0.548859,0.660620,0.359394,0.423847,4.0,0.320441,0.320468,-0.000027,0.482153,0.517847
val,346.0,0.271676,0.780347,0.887283,0.508632,0.596809,2.0,0.327889,0.328216,-0.000327,0.427746,0.572254
test,347.0,0.268012,0.812680,0.887608,0.511793,0.601832,2.0,0.323146,0.323367,-0.000221,0.443804,0.556196
v_bar,700.0,0.230000,0.672857,0.775714,0.432118,0.510157,3.0,0.333595,0.333735,-0.000140,0.467143,0.532857
h_bar,700.0,0.245714,0.687143,0.788571,0.453508,0.529646,2.0,0.324462,0.324500,-0.000038,0.480000,0.520000
line,700.0,0.192857,0.575714,0.692857,0.378653,0.446503,4.0,0.319545,0.319722,-0.000177,0.454286,0.545714
pie,302.0,0.115894,0.447020,0.543046,0.274131,0.328713,7.5,0.294349,0.294305,0.000045,0.480132,0.519868


Corrected metrics saved: /home/pnola130/projects/chart_data_retrieval/reports/metrics/clip_chart_table_metrics.json
